In [ ]:
import torch
import torchvision
print(torchvision.__version__)

In [ ]:
!pip install torch==2.8.0 torchvision==0.23.0 "torch_xla[tpu]==2.8.1" -f https://storage.googleapis.com/libtpu-releases/index.html

In [ ]:
import torch_xla
import torch_xla.core.xla_model as xm
import torch_xla.runtime as xr

device = xm.xla_device()
print("Device:", device)
print("Device type:", xr.device_type())

In [ ]:
import os
for root, dirs, files in os.walk("/content/dataset"):
    if files:
        print(root, "->", len(files), "images")

In [ ]:
!unzip -q /content/cats_dogs_light.zip -d /content/dataset

In [ ]:
import os
import random
import matplotlib.pyplot as plt
from PIL import Image

folder = "/content/dataset/cats_dogs_light/train"
files = random.sample(os.listdir(folder), 8)

plt.figure(figsize=(10, 5))
for i, name in enumerate(files):
    plt.subplot(2, 4, i + 1)
    plt.imshow(Image.open(os.path.join(folder, name)))
    plt.title("cat" if "cat" in name.lower() else "dog")
    plt.axis("off")
plt.show()

In [ ]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
import torch_xla.core.xla_model as xm
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

In [ ]:
class CatDogDataset(Dataset):
    def __init__(self, folder, transform=None):
        self.folder = folder
        self.files = sorted(os.listdir(folder))
        self.transform = transform
        self.classes = ["cat", "dog"]

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        name = self.files[idx]
        img = Image.open(os.path.join(self.folder, name)).convert("RGB")
        label = 0 if "cat" in name.lower() else 1
        if self.transform:
            img = self.transform(img)
        return img, label

In [ ]:
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
])

In [ ]:
base = "/content/dataset/cats_dogs_light"
train_data = CatDogDataset(f"{base}/train", transform)
test_data = CatDogDataset(f"{base}/test", transform)

print("Train:", len(train_data), "Test:", len(test_data))

In [ ]:
train_loader = DataLoader(train_data, batch_size=32, shuffle=True)
test_loader = DataLoader(test_data, batch_size=32, shuffle=False)

In [ ]:
import torch.nn as nn
from torchvision import models

model = models.resnet18(weights="DEFAULT")   # pretrained model
model.fc = nn.Linear(model.fc.in_features, 2)  # 2 classes: cat, dog
model = model.to(device)

In [ ]:
import torch.nn as nn

class CatDogCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 16, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(16, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.AdaptiveAvgPool2d((4, 4)),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 4 * 4, 128), nn.ReLU(),
            nn.Linear(128, 2),
        )

    def forward(self, x):
        return self.classifier(self.features(x))

In [ ]:
model = CatDogCNN().to(device)
print(model)

In [ ]:
import torch.optim as optim

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

In [ ]:
import os
import torch

epochs = 17   # total epochs
checkpoint_path = "/content/checkpoint.pth"

if os.path.exists(checkpoint_path):
    checkpoint = torch.load(checkpoint_path, map_location="cpu")
    model.load_state_dict(checkpoint["model_state"])
    optimizer.load_state_dict(checkpoint["optimizer_state"])

    start_epoch = checkpoint["epoch"]          # last saved epoch
    train_losses = checkpoint["train_losses"]
    train_accs = checkpoint["train_accs"]

    print("Checkpoint mila. Resuming from epoch:", start_epoch + 1, flush=True)
else:
    start_epoch = 0
    train_losses = []
    train_accs = []

    print("Checkpoint nahi mila. Epoch 1 se start ho raha hai.", flush=True)

In [ ]:
import time

for epoch in range(start_epoch, epochs):
    t0 = time.time()
    model.train()
    running_loss = 0
    correct = 0
    total = 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        xm.optimizer_step(optimizer)

        running_loss += loss.item() * images.size(0)
        correct += (outputs.argmax(1) == labels).sum().item()
        total += labels.size(0)

    epoch_loss = running_loss / total
    epoch_acc = correct / total
    train_losses.append(epoch_loss)
    train_accs.append(epoch_acc)

    xm.save({
        "epoch": epoch + 1,
        "model_state": model.state_dict(),
        "optimizer_state": optimizer.state_dict(),
        "train_losses": train_losses,
        "train_accs": train_accs,
    }, checkpoint_path)

    print(f"Epoch {epoch+1}/{epochs} - Loss: {epoch_loss:.4f} - Acc: {epoch_acc:.4f} - Time: {time.time()-t0:.0f}s - Checkpoint saved", flush=True)